<a href="https://colab.research.google.com/github/rizwanmahnoor1905-web/Machine-learning-track/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rizwanmahnoor1905-web/Machine-learning-track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Those client ids who donot have search or analytics access ask them to connect GSC and GA4.

| access_profile                  | Action                            | Reason Code                  |
| ------------------------------- | --------------------------------- | ---------------------------- |
| `no_search_or_analytics_access` | Ask client to connect GSC and GA4 | `NO_SEARCH_ANALYTICS_ACCESS` |
| `gsc_only`                      | Ask client to connect GA4         | `GA4_MISSING`                |
| `ga4_only`                      | Ask client to connect GSC         | `GSC_MISSING`                |
| `gsc_and_ga4`                   | No action needed                  | `FULL_ACCESS`                |


In [3]:
import duckdb
from google.colab import userdata

# Read the token securely from Colab Secrets
hf_token = userdata.get("Hftoken")

if not hf_token:
    raise ValueError("The Colab Secret named 'Hftoken' was not found.")

# Create the DuckDB connection
con = duckdb.connect()

# Give DuckDB permission to use the Hugging Face token
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

print("DuckDB is authenticated with Hugging Face.")

DuckDB is authenticated with Hugging Face.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)
sample = con.sql(f"""
SELECT *
FROM read_parquet('{march_path}')
LIMIT 20
""").df()
for col in sample.columns:
    print(col)

print(sample.columns)
top20 = con.sql(f"""
SELECT
    client_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,

    CASE
        WHEN gsc_impressions >= 1000 AND gsc_clicks < 20 THEN 100
        WHEN gsc_impressions >= 500 AND gsc_clicks < 20 THEN 75
        ELSE 25
    END AS score,

    CASE
        WHEN gsc_impressions >= 1000 AND gsc_clicks < 20
            THEN 'LOW_CTR_HIGH_IMPRESSIONS'
        WHEN gsc_impressions >= 500 AND gsc_clicks < 20
            THEN 'LOW_CTR_MEDIUM_IMPRESSIONS'
        ELSE 'NORMAL'
    END AS reason_code,

    CASE
        WHEN gsc_impressions >= 500 AND gsc_clicks < 20
            THEN 'Review title and meta description'
        ELSE 'No action required'
    END AS action

FROM read_parquet('{march_path}')

ORDER BY score DESC

LIMIT 20
""").df()

top20

top20

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month
Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_oth

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,report_date,gsc_impressions,gsc_clicks,score,reason_code,action
0,client_73cda7b4e4f265ea,2026-03-01,1082,2,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
1,client_73cda7b4e4f265ea,2026-03-01,2105,12,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
2,client_73cda7b4e4f265ea,2026-03-01,1074,0,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
3,client_73cda7b4e4f265ea,2026-03-01,1224,4,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
4,client_73cda7b4e4f265ea,2026-03-01,1692,0,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
5,client_73cda7b4e4f265ea,2026-03-01,1259,14,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
6,client_73cda7b4e4f265ea,2026-03-01,1314,4,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
7,client_73cda7b4e4f265ea,2026-03-01,1146,1,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
8,client_73cda7b4e4f265ea,2026-03-01,2442,4,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description
9,client_73cda7b4e4f265ea,2026-03-01,1806,0,100,LOW_CTR_HIGH_IMPRESSIONS,Review title and meta description


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

top20 = con.sql(f"""
SELECT

    client_hash_id,

    client_has_gsc,
    client_has_ga4,

    CASE

        WHEN client_has_gsc = FALSE
         AND client_has_ga4 = FALSE THEN 100

        WHEN client_has_gsc = FALSE
         AND client_has_ga4 = TRUE THEN 50

        WHEN client_has_gsc = TRUE
         AND client_has_ga4 = FALSE THEN 50

        ELSE 0

    END AS score,

    CASE

        WHEN client_has_gsc = FALSE
         AND client_has_ga4 = FALSE
            THEN 'NO_SEARCH_ANALYTICS_ACCESS'

        WHEN client_has_gsc = FALSE
         AND client_has_ga4 = TRUE
            THEN 'GSC_MISSING'

        WHEN client_has_gsc = TRUE
         AND client_has_ga4 = FALSE
            THEN 'GA4_MISSING'

        ELSE 'FULL_ACCESS'

    END AS reason_code,

    CASE

        WHEN client_has_gsc = FALSE
         AND client_has_ga4 = FALSE
            THEN 'Connect GSC and GA4'

        WHEN client_has_gsc = FALSE
         AND client_has_ga4 = TRUE
            THEN 'Connect GSC'

        WHEN client_has_gsc = TRUE
         AND client_has_ga4 = FALSE
            THEN 'Connect GA4'

        ELSE 'No action required'

    END AS action

FROM read_parquet('{march_path}')

ORDER BY score DESC

LIMIT 20

""").df()


top20["confidence_note"] = top20.apply(
    lambda row:
    "High - both connections missing."
    if row["score"] == 100
    else "Medium - only one connection is missing."
    if row["score"] == 50
    else "Low - no action required.",
    axis=1
)


top20["what_would_make_it_wrong"] = (
    "The client may have recently connected GSC or GA4, or the dataset may not yet reflect the latest status."
)

top20

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,client_has_gsc,client_has_ga4,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
1,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
2,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
3,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
4,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
5,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
6,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
7,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
8,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...
9,client_73cda7b4e4f265ea,True,False,50,GA4_MISSING,Connect GA4,Medium - only one connection is missing.,The client may have recently connected GSC or ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks + leakage check

Weak Picks

The weakest picks are clients that are flagged because they appear to have missing Google Search Console (GSC) or Google Analytics 4 (GA4) access. These recommendations could be incorrect if:

The client recently connected GSC or GA4, but the dataset has not yet been updated.
The client intentionally does not want to connect one or both services.
The missing access is temporary because of a permission or synchronization issue rather than a permanent lack of access.

Therefore, these cases should be manually verified before taking action.
Leakage Check

My rule only uses information that is available in the current dataset:

client_has_gsc
client_has_ga4

It does not use:

future performance data,
future dates,
product-generated flags,
labels created after the fact.

Therefore, there is no future-window leakage or label leakage in this baseline rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.